# 🏗️ Document Processing Pipeline

**Day 6 — Notebook 4 of 4 | Estimated Time: 35 minutes**

---

## What You'll Learn
- Build a complete end-to-end document processing pipeline
- Load and parse multiple document formats (plain text, structured data)
- Enrich chunks with metadata (source, position, date)
- Persist a ChromaDB collection and query it
- Handle duplicate detection and incremental updates

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" chromadb langchain-text-splitters

In [ ]:
import sys, os, hashlib, json
from datetime import datetime
import numpy as np
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
EMBEDDING_MODEL = "text-embedding-004"
GEN_MODEL = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. The Pipeline Architecture

A document processing pipeline has four stages:

```
┌────────────┐    ┌────────────┐    ┌────────────┐    ┌────────────────┐
│   LOAD     │ →  │   CHUNK    │ →  │   EMBED    │ →  │     STORE      │
│            │    │            │    │            │    │                │
│ Raw text   │    │ Splits     │    │ Vectors    │    │ ChromaDB       │
│ Files      │    │ into       │    │ 768-dim    │    │ + metadata     │
│ URLs       │    │ passages   │    │ floats     │    │ + original text│
└────────────┘    └────────────┘    └────────────┘    └────────────────┘
```

At query time:
```
User Query → Embed → Search ChromaDB → Retrieve top-k chunks → Feed to LLM → Answer
```

---

## 2. Stage 1: Load

Real documents come in many formats. Here we handle plain text files and simulated structured data. (PDF loading comes in Level 2.)

In [ ]:
# Create sample documents on disk to simulate loading real files
import tempfile, pathlib

DOCS_DIR = pathlib.Path("/tmp/day6_docs")
DOCS_DIR.mkdir(exist_ok=True)

sample_files = {
    "python_guide.txt": """
Python Programming Guide
========================

Python is a high-level, interpreted programming language known for its clean syntax and 
readability. Created by Guido van Rossum and first released in 1991, Python has grown to 
become one of the most popular programming languages in the world.

Variables in Python are dynamically typed — you don't need to declare a type before using 
a variable. Simply assign a value and Python infers the type. For example: x = 42 creates 
an integer, while name = "Alice" creates a string.

Python supports multiple programming paradigms including procedural, object-oriented, and 
functional programming. This flexibility makes it suitable for a wide range of applications 
from web development to data science to automation.

The Python Package Index (PyPI) hosts over 400,000 packages. You can install any package 
using pip: pip install package-name. Virtual environments (venv) isolate dependencies 
between projects to avoid conflicts.

List comprehensions are a concise way to create lists: [x**2 for x in range(10)] generates 
a list of squares. Dictionary comprehensions and generator expressions follow similar syntax.
""",
    "git_guide.txt": """
Git Version Control Guide
=========================

Git is a distributed version control system that tracks changes in files over time. 
Created by Linus Torvalds in 2005, Git has become the standard tool for source code management.

The three main areas in Git are the working directory, staging area, and repository. 
Changes start in the working directory. You stage them with git add and commit them 
to the repository with git commit.

Branches let you work on features independently from the main codebase. Create a branch 
with git checkout -b feature-name. Switch between branches using git checkout branch-name. 
Merge completed work back with git merge.

Remote repositories like GitHub or GitLab let teams collaborate. Push your changes with 
git push origin branch-name. Pull the latest changes from the remote with git pull.

A good commit message describes WHY a change was made, not just what was changed. 
Follow the convention: short summary (50 chars), blank line, detailed body if needed.
""",
    "docker_guide.txt": """
Docker Containerisation Guide
==============================

Docker is a platform for building, shipping, and running applications in containers. 
Containers package an application and its dependencies into a single portable unit that 
runs consistently across different environments.

A Dockerfile defines how to build a Docker image. It specifies the base image, copies 
application code, installs dependencies, and defines the start command. Build an image 
with docker build -t my-app:latest .

Docker Compose is used to define and run multi-container applications. A docker-compose.yml 
file specifies services, networks, and volumes. Start all services with docker-compose up. 
Stop them with docker-compose down.

Container networking allows containers to communicate with each other. Docker creates a 
default bridge network. Named networks give you more control over how containers connect. 
Expose ports with the -p flag: docker run -p 8080:80 my-app.

Volumes persist data beyond container lifecycles. Mount a host directory with 
-v /host/path:/container/path. Named volumes are managed by Docker and are more portable.
""",
}

# Write files to disk
for filename, content in sample_files.items():
    (DOCS_DIR / filename).write_text(content)

print(f"Created {len(sample_files)} sample documents in {DOCS_DIR}")
for f in DOCS_DIR.iterdir():
    print(f"  {f.name} ({f.stat().st_size} bytes)")

In [ ]:
def load_text_file(filepath: str) -> dict:
    """Load a text file and return content + metadata."""
    path = pathlib.Path(filepath)
    content = path.read_text(encoding="utf-8")
    
    # Generate a stable hash to detect duplicates
    content_hash = hashlib.md5(content.encode()).hexdigest()[:8]
    
    return {
        "content": content.strip(),
        "source": path.name,
        "file_path": str(path),
        "content_hash": content_hash,
        "loaded_at": datetime.utcnow().isoformat() + "Z",
    }


# Load all documents
raw_docs = [load_text_file(f) for f in sorted(DOCS_DIR.glob("*.txt"))]
print(f"Loaded {len(raw_docs)} documents")
for doc in raw_docs:
    print(f"  {doc['source']}: {len(doc['content'])} chars, hash={doc['content_hash']}")

---

## 3. Stage 2: Chunk

In [ ]:
def chunk_document(doc: dict, chunk_size: int = 500, chunk_overlap: int = 80) -> list[dict]:
    """
    Split a document into chunks, preserving source metadata on every chunk.
    Returns a list of chunk dicts ready for embedding.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    
    text_chunks = splitter.split_text(doc["content"])
    
    chunks = []
    for i, chunk_text in enumerate(text_chunks):
        # Stable chunk ID: hash of (source + chunk_index + content)
        chunk_id = hashlib.md5(f"{doc['source']}_{i}_{chunk_text}".encode()).hexdigest()[:12]
        
        chunks.append({
            "id": chunk_id,
            "text": chunk_text,
            "source": doc["source"],
            "chunk_index": i,
            "total_chunks": len(text_chunks),
            "doc_hash": doc["content_hash"],
        })
    
    return chunks


# Chunk all documents
all_chunks = []
for doc in raw_docs:
    doc_chunks = chunk_document(doc)
    all_chunks.extend(doc_chunks)
    print(f"  {doc['source']}: {len(doc_chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

---

## 4. Stage 3: Embed

Embed all chunks in batches (to stay within API limits).

In [ ]:
def embed_batch(texts: list[str], batch_size: int = 100) -> list[list[float]]:
    """
    Embed texts in batches to respect API limits.
    Gemini allows up to 100 texts per batch.
    """
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        result = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=batch,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
        )
        all_embeddings.extend([e.values for e in result.embeddings])
        print(f"  Embedded batch {i//batch_size + 1}: {len(batch)} texts")
    return all_embeddings


print("Embedding all chunks...")
texts_to_embed = [c["text"] for c in all_chunks]
embeddings = embed_batch(texts_to_embed)
print(f"\n✅ Generated {len(embeddings)} embeddings ({len(embeddings[0])} dims each)")

---

## 5. Stage 4: Store

In [ ]:
# Set up persistent ChromaDB
DB_PATH = "/tmp/day6_pipeline_db"
os.makedirs(DB_PATH, exist_ok=True)

chroma = chromadb.PersistentClient(path=DB_PATH)

# Delete if exists (clean run)
try:
    chroma.delete_collection("dev_docs")
except Exception:
    pass

collection = chroma.create_collection(
    name="dev_docs",
    metadata={"hnsw:space": "cosine"},
)

# Add all chunks to ChromaDB
collection.upsert(
    ids=[c["id"] for c in all_chunks],
    documents=[c["text"] for c in all_chunks],
    embeddings=embeddings,
    metadatas=[
        {
            "source": c["source"],
            "chunk_index": c["chunk_index"],
            "total_chunks": c["total_chunks"],
            "doc_hash": c["doc_hash"],
        }
        for c in all_chunks
    ],
)

print(f"✅ Pipeline complete! Collection: {collection.count()} chunks stored")

---

## 6. Query the Pipeline

In [ ]:
def query_knowledge_base(query: str, n_results: int = 3, source_filter: str = None) -> list[dict]:
    """Query the document store."""
    # Embed the query
    q_result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    )
    q_vec = q_result.embeddings[0].values
    
    # Build query kwargs
    kwargs = {
        "query_embeddings": [q_vec],
        "n_results": n_results,
        "include": ["documents", "metadatas", "distances"],
    }
    if source_filter:
        kwargs["where"] = {"source": source_filter}
    
    results = collection.query(**kwargs)
    
    return [
        {
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"],
            "chunk_index": results["metadatas"][0][i]["chunk_index"],
            "score": 1 - results["distances"][0][i],
        }
        for i in range(len(results["ids"][0]))
    ]


# Test queries
queries = [
    "How do I install a Python package?",
    "What is the difference between staging and committing in Git?",
    "How do containers share data between restarts?",
    "How do I run multiple containers together?",
]

for q in queries:
    print(f"🔍 {q}")
    results = query_knowledge_base(q, n_results=2)
    for r in results:
        print(f"   [{r['score']:.3f}] ({r['source']}, chunk {r['chunk_index']}) {r['text'][:90]}...")
    print()

---

## 7. Retrieval + Generation (Full RAG Preview)

In [ ]:
def answer_with_docs(question: str, n_results: int = 3) -> None:
    """Retrieve relevant chunks and generate a grounded answer."""
    # Retrieve
    chunks = query_knowledge_base(question, n_results=n_results)
    
    if not chunks or chunks[0]["score"] < 0.4:
        print("❌ No relevant documents found.")
        return
    
    # Build context from retrieved chunks
    context_parts = []
    sources = set()
    for chunk in chunks:
        context_parts.append(f"[Source: {chunk['source']}]\n{chunk['text']}")
        sources.add(chunk["source"])
    context = "\n\n---\n\n".join(context_parts)
    
    # Generate
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Answer the question using only the provided documentation excerpts.
Be concise and cite the source document for each key point.

<documentation>
{context}
</documentation>

Question: {question}

Answer:""",
        config=types.GenerateContentConfig(temperature=0.1),
    )
    
    print(f"❓ {question}")
    print(f"📚 Sources used: {', '.join(sources)}")
    print(f"🤖 {response.text.strip()}")
    print()


answer_with_docs("How do I collaborate with teammates using Git?")
answer_with_docs("What are Docker volumes and why do I need them?")
answer_with_docs("How do list comprehensions work in Python?")

---

## 8. Incremental Updates

In production, documents change. The pipeline should only re-process changed files.

In [ ]:
def get_indexed_hashes(collection) -> dict:
    """Get the content hash for every source document already in the index."""
    all_meta = collection.get(include=["metadatas"])["metadatas"]
    hashes = {}
    for meta in all_meta:
        source = meta.get("source")
        doc_hash = meta.get("doc_hash")
        if source and doc_hash:
            hashes[source] = doc_hash
    return hashes


def incremental_update(new_docs: list[dict], collection) -> dict:
    """Only process documents that are new or changed (hash changed)."""
    indexed = get_indexed_hashes(collection)
    
    to_process = []
    for doc in new_docs:
        source = doc["source"]
        if source not in indexed:
            print(f"  ➕ New document: {source}")
            to_process.append(doc)
        elif indexed[source] != doc["content_hash"]:
            print(f"  🔄 Changed document: {source} (hash changed)")
            to_process.append(doc)
        else:
            print(f"  ✅ Unchanged: {source} (skipped)")
    
    return {"total": len(new_docs), "processed": len(to_process), "skipped": len(new_docs) - len(to_process)}


# Simulate: run pipeline again — all docs should be skipped (unchanged)
print("Incremental update check:")
stats = incremental_update(raw_docs, collection)
print(f"\n  Total: {stats['total']}, Processed: {stats['processed']}, Skipped: {stats['skipped']}")

In [ ]:
# Simulate a document change
updated_doc = load_text_file(DOCS_DIR / "python_guide.txt")
# Modify the content to simulate a file change
updated_doc["content"] += "\n\nPython 3.13 was released in 2024 with significant performance improvements."
updated_doc["content_hash"] = hashlib.md5(updated_doc["content"].encode()).hexdigest()[:8]

print("After modifying python_guide.txt:")
stats = incremental_update([updated_doc] + raw_docs[1:], collection)
print(f"\n  Total: {stats['total']}, Processed: {stats['processed']}, Skipped: {stats['skipped']}")

---

## 🏋️ Exercise 1: Add a New Document Format

Extend the pipeline to handle Markdown files with section-aware chunking.

In [ ]:
# Exercise 1: Markdown-aware loading

sample_markdown = """
# API Reference

## Authentication
All requests must include an `Authorization` header with a Bearer token.
Tokens expire after 24 hours. Request a new token at `/auth/token`.

## Endpoints

### GET /users
Returns a paginated list of all users. Supports `?page=N&limit=N` query params.
Requires `admin` role.

### POST /users
Creates a new user. Request body must include `email` and `name` fields.
Returns the created user with an auto-generated `id`.

### DELETE /users/{id}
Soft-deletes the user with the given ID. The user is marked as inactive but
not removed from the database. Requires `admin` role.

## Error Codes
- 400: Bad Request — missing or invalid parameters
- 401: Unauthorized — missing or expired token
- 403: Forbidden — insufficient permissions
- 404: Not Found — resource does not exist
- 500: Internal Server Error — contact support
"""

# TODO:
# 1. Write this markdown to /tmp/day6_docs/api_reference.md
# 2. Build a load_markdown_file() function that splits on '##' headers
#    Each section (header + its text) becomes one chunk
# 3. Add the chunks to the collection
# 4. Test with: query_knowledge_base("What permissions do I need to delete a user?")

def load_markdown_by_section(content: str) -> list[str]:
    """
    Split markdown into sections at ## headings.
    Each section includes its heading and body.
    """
    # TODO: Implement this
    # Hint: re.split(r'(?=^## )', content, flags=re.MULTILINE)
    pass


# Test
# sections = load_markdown_by_section(sample_markdown)
# for s in sections:
#     print(f"Section ({len(s)} chars): {s[:60]}...")

---

## 🏋️ Exercise 2: Build Your Own Pipeline

Put everything together to build a pipeline for your own documents.

In [ ]:
# Exercise 2: Full pipeline for your own corpus
# TODO:
# 1. Choose 3-5 topics you care about
# 2. Write a short (150-300 word) document on each topic
# 3. Run the full pipeline: load → chunk → embed → store
# 4. Ask 5 questions and evaluate the answers
# 5. Bonus: Try two chunk sizes and compare retrieval quality

my_docs_dir = pathlib.Path("/tmp/my_docs")
my_docs_dir.mkdir(exist_ok=True)

# TODO: Write your documents
# (my_docs_dir / "topic1.txt").write_text("...")

# TODO: Run the pipeline
# TODO: Query and evaluate

---

## Pipeline Summary

```python
# The complete pipeline in ~20 lines:

# 1. LOAD
docs = [load_text_file(f) for f in doc_dir.glob("*.txt")]

# 2. CHUNK
chunks = []
for doc in docs:
    chunks.extend(chunk_document(doc, chunk_size=500, chunk_overlap=80))

# 3. EMBED
embeddings = embed_batch([c["text"] for c in chunks])

# 4. STORE
collection.upsert(
    ids=[c["id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    embeddings=embeddings,
    metadatas=[{"source": c["source"], ...} for c in chunks],
)
```

---

## Day 6 Complete! 🎉

You've learned:
- ✅ Why vector databases exist and how HNSW indexing works (Notebook 1)
- ✅ Full ChromaDB CRUD: create, add, query, update, delete, persist (Notebook 2)
- ✅ Fixed-size, recursive, sentence-aware, and code chunking strategies (Notebook 3)
- ✅ End-to-end document processing pipeline with incremental updates (Notebook 4)

**Next:** Day 7 — Retrieval Augmented Generation (RAG)

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| ChromaDB Docs | Docs | [docs.trychroma.com](https://docs.trychroma.com/) |
| LangChain Text Splitters | Docs | [python.langchain.com/docs/modules/data_connection/document_transformers](https://python.langchain.com/docs/modules/data_connection/document_transformers/) |
| Chunking for RAG | Blog | [pinecone.io/learn/chunking-strategies](https://www.pinecone.io/learn/chunking-strategies/) |